# BaIP: train one configuration on Colab

Runtime > Change runtime type > **L4 GPU** (or A100). T4 also works but is about 2x slower.

One-time setup on Google Drive:
```
MyDrive/BaIP/data/Images/            S###_k.jpg
MyDrive/BaIP/data/Images-2/          S###_k.jpg (the 79 extra files)
MyDrive/BaIP/data/labels20260217_sort3d_hsi_measurements.xlsx
```
Results go to `MyDrive/BaIP/results/<run>/` (config, split, history, metrics, predictions) and one line per run in `MyDrive/BaIP/results/runs.csv`. Checkpoints go to `MyDrive/BaIP/checkpoints/`.

In [ ]:
BRANCH = 'baip-pipeline' # git branch to run
EXPERIMENT = 'crop448_holdout'   # repro_elsherif | resize448_holdout | crop448_holdout
OVERRIDES = []           # e.g. ['train.lr=1e-4', 'data.augment=true', 'split.scheme=cv', 'split.fold=0']

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/repo'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/merijnmolenaar/Textile_Classification_CNN_Merijn.git {REPO}
!cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q && git log -1 --format='commit %h  %s'
!pip -q install pyyaml openpyxl

In [ ]:
# Check the data is where paths_colab.yaml expects it
for p in ['/content/drive/MyDrive/BaIP/data/Images', '/content/drive/MyDrive/BaIP/data/Images-2']:
    print(p, len(os.listdir(p)) if os.path.isdir(p) else 'MISSING')
print('labels:', os.path.exists('/content/drive/MyDrive/BaIP/data/labels20260217_sort3d_hsi_measurements.xlsx'))

**Smoke test** (about 2 min, includes copying images to local disk): 2 epochs of 3 batches. Writes to `results/smoke`. Skip once it has worked.

In [ ]:
!cd {REPO} && python scripts/train.py configs/{EXPERIMENT}.yaml --paths configs/paths_colab.yaml \
    --set train.epochs=2 --set train.max_batches=3 --set name=smoke_{EXPERIMENT} \
    --set out_dir=/content/drive/MyDrive/BaIP/results/smoke

**Full run.** The first epoch prints an estimate of the total time. Keep the tab open (or use Colab Pro background execution).

In [ ]:
ov = ' '.join(f'--set {o}' for o in OVERRIDES)
!cd {REPO} && python scripts/train.py configs/{EXPERIMENT}.yaml --paths configs/paths_colab.yaml {ov}

In [ ]:
import pandas as pd
pd.read_csv('/content/drive/MyDrive/BaIP/results/runs.csv').tail(10)